### Data Preview 

In [0]:
%sql --Customer Table 
SELECT * FROM workspace.bronze.customers;

In [0]:
%sql --Product Table 
SELECT * FROM workspace.bronze.products;

In [0]:
%sql
--Sales Table
SELECT * FROM workspace.bronze.sales;

### Total Rows In Each Table

In [0]:
%sql
SELECT 'sales'     AS table_name, COUNT(*) AS row_count FROM workspace.bronze.sales
UNION ALL
SELECT 'customers'  AS table_name, COUNT(*) AS row_count FROM workspace.bronze.customers
UNION ALL
SELECT 'products'   AS table_name, COUNT(*) AS row_count FROM workspace.bronze.products;

### NULL Checks

In [0]:
%sql
--NULL CHECK FOR CUSTOMERS
SELECT
  COUNT(*)                                                    AS total_rows,
  SUM(CASE WHEN first_name     IS NULL THEN 1 ELSE 0 END)    AS null_first_name,
  SUM(CASE WHEN last_name      IS NULL THEN 1 ELSE 0 END)    AS null_last_name,
  SUM(CASE WHEN country        IS NULL THEN 1 ELSE 0 END)    AS null_country,
  SUM(CASE WHEN gender         IS NULL THEN 1 ELSE 0 END)    AS null_gender,
  SUM(CASE WHEN birthdate      IS NULL THEN 1 ELSE 0 END)    AS null_birthdate,
  SUM(CASE WHEN marital_status IS NULL THEN 1 ELSE 0 END)    AS null_marital_status
FROM workspace.bronze.customers;


In [0]:
%sql
--NULL CHECK FOR PRODUCTS
SELECT
  COUNT(*)                                                  AS total_rows,
  SUM(CASE WHEN product_name IS NULL THEN 1 ELSE 0 END)    AS null_product_name,
  SUM(CASE WHEN category     IS NULL THEN 1 ELSE 0 END)    AS null_category,
  SUM(CASE WHEN subcategory  IS NULL THEN 1 ELSE 0 END)    AS null_subcategory,
  SUM(CASE WHEN product_line IS NULL THEN 1 ELSE 0 END)    AS null_product_line,
  SUM(CASE WHEN cost         IS NULL THEN 1 ELSE 0 END)    AS null_cost
FROM workspace.bronze.products;


In [0]:
%sql
--NULL CHECK FOR SALES
SELECT
  COUNT(*)                                                  AS total_rows,
  SUM(CASE WHEN order_number   IS NULL THEN 1 ELSE 0 END)  AS null_order_number,
  SUM(CASE WHEN product_key    IS NULL THEN 1 ELSE 0 END)  AS null_product_key,
  SUM(CASE WHEN customer_key   IS NULL THEN 1 ELSE 0 END)  AS null_customer_key,
  SUM(CASE WHEN order_date     IS NULL THEN 1 ELSE 0 END)  AS null_order_date,
  SUM(CASE WHEN shipping_date  IS NULL THEN 1 ELSE 0 END)  AS null_shipping_date,
  SUM(CASE WHEN sales_amount   IS NULL THEN 1 ELSE 0 END)  AS null_sales_amount,
  SUM(CASE WHEN quantity       IS NULL THEN 1 ELSE 0 END)  AS null_quantity,
  SUM(CASE WHEN price          IS NULL THEN 1 ELSE 0 END)  AS null_price
FROM workspace.bronze.sales;


### Duplicate Check

In [0]:
%sql
--Are There Duplicate Customers?
SELECT
  customer_id,
  COUNT(*) AS duplicate_count
FROM workspace.bronze.customers
GROUP BY customer_id
HAVING COUNT(*) > 1;


In [0]:
%sql
--Are There Duplicate Peoducts?
SELECT
  product_id,
  COUNT(*) AS duplicate_count
FROM workspace.bronze.products
GROUP BY product_id
HAVING COUNT(*) > 1;


In [0]:
%sql
--Are There Duplicate Orders? 
SELECT
  order_number,
  COUNT(*) AS duplicate_count
FROM workspace.bronze.sales
GROUP BY order_number
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC;


### Category Check 
 Understanding our Categorical Columns



In [0]:
%sql
--What Countries Exist in the Data? 
SELECT
  country,
  COUNT(*) AS customer_count
FROM workspace.bronze.customers
GROUP BY country
ORDER BY customer_count DESC;


In [0]:
%sql
--What Gender Exist in the Data?
SELECT
  gender,
  COUNT(*) AS count
FROM workspace.bronze.customers
GROUP BY gender;


In [0]:
%sql
--What Product Catregories AND Subcategories Exist?
SELECT
  category,
  subcategory,
  COUNT(*) AS product_count
FROM workspace.bronze.products
GROUP BY category, subcategory
ORDER BY category;


In [0]:
%sql
--What Product Lines Exist?
SELECT
  product_line,
  COUNT(*) AS count
FROM workspace.bronze.products
GROUP BY product_line;

### Numeric Check
Understanding our Numeric Columns

In [0]:
%sql
--Sales Amount Health Check
-- Min should not be negative. Max should not be unrealistically huge.
SELECT
  MIN(sales_amount)   AS min_sales,
  MAX(sales_amount)   AS max_sales,
  AVG(sales_amount)   AS avg_sales,
  SUM(sales_amount)   AS total_revenue,
 
  MIN(quantity)       AS min_quantity,
  MAX(quantity)       AS max_quantity,
  AVG(quantity)       AS avg_quantity,
 
  MIN(price)          AS min_price,
  MAX(price)          AS max_price
FROM workspace.bronze.sales;

In [0]:
%sql
--CHECKING FOR ROWS WHERE PRICE × QUANTITY DOES NOT EQUAL SALES_AMOUNT
-- These are data integrity errors
SELECT
  order_number,
  price,
  quantity,
  sales_amount,
  ROUND(price * quantity, 2)  AS expected_amount
FROM workspace.bronze.sales
WHERE ROUND(price * quantity, 2) != ROUND(sales_amount, 2)


In [0]:
%sql
--Product Cost Health Check
SELECT
  MIN(cost)   AS min_cost,
  MAX(cost)   AS max_cost,
  AVG(cost)   AS avg_cost
FROM workspace.bronze.products


In [0]:
%sql
--Investigating to understand why cost is zero
SELECT *
FROM workspace.bronze.products
WHERE cost = 0;

### Date Check
Understanding our Date Columns

In [0]:
%sql
--What is the Date Range of the Sales Data?
SELECT
  MIN(order_date)      AS earliest_order,
  MAX(order_date)      AS latest_order,
  MIN(shipping_date)   AS earliest_ship,
  MAX(shipping_date)   AS latest_ship
FROM workspace.bronze.sales;

In [0]:
%sql
--HOW MANY ORDERS AND HOW MUCH REVENUE PER YEAR?
SELECT
  YEAR(CAST(order_date AS DATE))  AS order_year,
  COUNT(*)                         AS total_orders,
  SUM(sales_amount)                AS total_revenue
FROM workspace.bronze.sales
GROUP BY order_year
ORDER BY order_year;